# 02 Preparing the data

This notebook makes the clean table that all the models use. It groups listings so related ones don't end up in different splits, removes personal data, makes the 70/15/15 split, creates the price levels from the training set only, and cleans the descriptions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# same folder setup as notebook 01
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/airbnb_p22")
except ImportError:
    PROJECT_DIR = Path("airbnb_p22")

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"   # small files that are safe to put on GitHub
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

try:
    raw = pd.read_csv(DATA_DIR / "raw" / "listings_2026-09-16.csv.gz",
                      compression="gzip", low_memory=False)
    photo_manifest = pd.read_csv(DATA_DIR / "photo_manifest.csv")
except (OSError, pd.errors.ParserError) as e:
    raise RuntimeError(f"Run notebook 01 first, couldn't load its files: {e}")
print(f"{len(raw):,} listings, {raw.shape[1]} columns")
print(f"{photo_manifest['ok'].sum():,} photos available")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
25,715 listings, 90 columns
18,802 photos available


## Grouping related listings

In the initial analysis we only kept exact copies of a description together. But one host with many listings usually reuses the same writing style, photos and prices, even when the text isn't an exact copy. If those listings were split between training and testing, the model could just recognise the host instead of learning what makes a listing expensive. This is the same kind of shortcut as the speaker shortcut in the Week 8 practical.

So two listings go in the same group if they have the same host or the same description, and a whole group always goes into the same split. My first run made one group of 1,582 listings (6.2%), because some generic template descriptions are used by lots of unrelated hosts and chained them all together. A text shared by many different hosts doesn't identify a host, so now only descriptions used by 3 hosts or fewer link listings.

host_id is only used here to build the groups and is never saved. All host columns and the exact location are dropped straight after, except host_name, which I keep for a moment to remove host names from the descriptions and then drop before saving.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

# Two listings go in the same group if they have the same host OR the same
# description. Chains count too: if A and B share a host and B and C share a
# description, all three end up together. connected_components finds these
# groups by treating listings, hosts and descriptions as nodes in one graph.
host_key = raw["host_id"].astype(str).where(raw["host_id"].notna(),
                                             "no_host_" + raw["id"].astype(str))
desc_key = raw["description"].fillna("").str.strip().str.lower()
# Very short texts like "Welcome!" can be shared by unrelated hosts, so only
# descriptions with at least 10 words are used to link listings
desc_key = desc_key.where(desc_key.str.split().str.len() >= 10)

# My first run made one group of 1,582 listings. Some descriptions are generic
# template text used by lots of unrelated hosts, and they chained everyone
# together. A text shared by many different hosts isn't a host fingerprint,
# so only texts used by 3 hosts or fewer link listings.
hosts_per_text = host_key.groupby(desc_key).nunique()
print(f"Descriptions shared by more than 3 hosts: {(hosts_per_text > 3).sum()}, "
      f"covering {desc_key.map(hosts_per_text).gt(3).mean():.1%} of listings")
desc_key = desc_key.where(desc_key.map(hosts_per_text) <= 3)

n = len(raw)
host_code = pd.factorize(host_key)[0]
desc_code, desc_uniques = pd.factorize(desc_key)   # -1 where there's no description
has_desc = desc_code >= 0
n_hosts = host_code.max() + 1

listing_nodes = np.concatenate([np.arange(n), np.arange(n)[has_desc]])
other_nodes = np.concatenate([n + host_code, n + n_hosts + desc_code[has_desc]])
size = n + n_hosts + len(desc_uniques)
graph = coo_matrix((np.ones(len(listing_nodes)), (listing_nodes, other_nodes)),
                   shape=(size, size))
_, labels = connected_components(graph, directed=False)
raw["group"] = labels[:n]

group_sizes = raw["group"].value_counts()
biggest = group_sizes.index[0]
print(f"{raw['host_id'].nunique():,} hosts, {len(group_sizes):,} groups")
print(f"Listings in a group of more than one: {(raw['group'].map(group_sizes) > 1).mean():.1%}")
print(f"Biggest group: {group_sizes.max()} listings ({group_sizes.max() / n:.1%}), "
      f"{host_key[raw['group'] == biggest].nunique()} hosts in it")
print(f"Biggest single host: {host_key.value_counts().max()} listings")
if group_sizes.max() > 0.05 * n:
    print("Warning: the biggest group has over 5% of listings, check before splitting")

# Privacy: now that the groups exist, drop every host column and the exact
# location. host_id was only used in memory for the grouping and is never saved.
# host_name stays for now only so it can be removed from the descriptions
# later, and it's dropped before saving.
keep = ["id", "group", "host_name", "description", "price", "room_type",
        "property_type", "accommodates", "bedrooms", "beds", "bathrooms_text",
        "amenities", "neighbourhood_cleansed", "number_of_reviews",
        "review_scores_rating"]
df = raw[keep].copy()
dropped = [c for c in raw.columns if c not in keep]
print(f"Dropped {len(dropped)} columns, including "
      f"{sum(c.startswith('host_') for c in dropped)} host columns and latitude/longitude")
del raw

Descriptions shared by more than 3 hosts: 15, covering 2.1% of listings
13,999 hosts, 13,925 groups
Listings in a group of more than one: 56.5%
Biggest group: 310 listings (1.2%), 1 hosts in it
Biggest single host: 310 listings
Dropped 76 columns, including 19 host columns and latitude/longitude


## Price and the 70/15/15 split

Listings without a price can't get a label, so they're removed. Then I split by group: 70% train, 15% validation, 15% test. Validation is for choosing settings and early stopping, and the test set is only used once at the end.

My first try used GroupShuffleSplit, but that splits by the number of groups, not listings, so it came out 72.8/14.4/12.8. Now I shuffle the groups and give them out in order until each split has its share of listings.

In [ ]:
df["price"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[$,]", "", regex=True),
                            errors="coerce")
print(f"No price: {df['price'].isna().mean():.1%}")
df = df[df["price"] > 0].reset_index(drop=True)
print(f"{len(df):,} listings with a price")

# GroupShuffleSplit splits by number of groups, not listings, so with a few big
# groups my first run came out 72.8/14.4/12.8. Instead I shuffle the groups and
# hand them out in order until each split has its share of listings.
rng = np.random.default_rng(SEED)
groups = df["group"].unique()
rng.shuffle(groups)
sizes = df["group"].value_counts().reindex(groups)
share_so_far = sizes.cumsum() / sizes.sum()
split_of_group = pd.Series(np.select([share_so_far <= 0.70, share_so_far <= 0.85],
                                     ["train", "val"], "test"), index=groups)
df["split"] = df["group"].map(split_of_group)

# no group should ever appear in two splits
assert df.groupby("group")["split"].nunique().max() == 1
print(df["split"].value_counts(normalize=True).round(3).to_dict())

No price: 26.1%
19,004 listings with a price
{'train': 0.695, 'val': 0.154, 'test': 0.151}


## Making the price levels

In the initial analysis the cap and the price levels were worked out on all listings. Here I only use the training set, so validation and test prices can't affect how the labels are made. The cap is the 99th percentile of training prices and it's applied to every split. The budget, mid-range and premium edges are the training tertiles, so the dollar cut-offs might move slightly from the $203 and $325 in our proposal.

In [ ]:
import json

train = df["split"] == "train"

# Cap and tertile edges come from the training set only, so the validation
# and test prices don't influence how the labels are made
cap = df.loc[train, "price"].quantile(0.99)
above = df["price"] > cap
print(f"Cap (train 99th percentile): ${cap:,.0f}")
print("Removed above the cap:", df.loc[above, "split"].value_counts().to_dict())
df = df[~above].reset_index(drop=True)
train = df["split"] == "train"

low, high = df.loc[train, "price"].quantile([1 / 3, 2 / 3])
df["price_level"] = pd.cut(df["price"], [-np.inf, low, high, np.inf],
                           labels=["budget", "mid-range", "premium"])
print(f"Budget up to ${low:,.0f}, mid-range up to ${high:,.0f}, premium above")

balance = pd.crosstab(df["split"], df["price_level"], normalize="index").round(3)
balance["listings"] = df["split"].value_counts()
print(balance.loc[["train", "val", "test"]])

try:
    with open(RESULTS_DIR / "target_edges.json", "w") as f:
        json.dump({"cap": cap, "budget_max": low, "midrange_max": high}, f, indent=2)
except OSError as e:
    print(f"Couldn't save target_edges.json: {e}")

Cap (train 99th percentile): $1,533
Removed above the cap: {'train': 133, 'test': 40, 'val': 23}
Budget up to $200, mid-range up to $323, premium above
price_level  budget  mid-range  premium  listings
split                                            
train         0.335      0.332    0.333     13082
val           0.311      0.328    0.360      2896
test          0.317      0.348    0.335      2830


## Checking the descriptions

Before cleaning I count what's actually in the text: HTML left over from the website, links, emails and phone numbers (personal details), and dollar amounts. Dollar amounts matter because a description saying "$150 a night" basically gives away the label.

In [ ]:
import re

desc = df["description"].fillna("")
checks = {
    "HTML tags": r"<[^>]+>",
    "HTML entities (&amp; etc.)": r"&[a-z]+;|&#\d+;",
    "links": r"https?://|www\.",
    "emails": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone numbers": r"(?:\+?61|0)[\s-]?\d(?:[\s-]?\d){7,9}",
    "dollar amounts": r"\$\s?\d",
}
for name, pattern in checks.items():
    print(f"{name}: {desc.str.contains(pattern, regex=True).mean():.1%} of descriptions")

n_words = desc.str.split().str.len()
print(f"Empty: {(desc.str.strip() == '').mean():.1%}")
print(f"Length: median {n_words.median():.0f} words, 95th percentile "
      f"{n_words.quantile(0.95):.0f}, max {n_words.max()}")
print(f"Exact copies: {desc[desc != ''].duplicated(keep=False).mean():.1%} of descriptions "
      "share their text with another listing")

HTML tags: 54.4% of descriptions
HTML entities (&amp; etc.): 0.7% of descriptions
links: 0.2% of descriptions
emails: 0.0% of descriptions
phone numbers: 0.0% of descriptions
dollar amounts: 1.6% of descriptions
Empty: 0.4%
Length: median 71 words, 95th percentile 90, max 188
Exact copies: 16.4% of descriptions share their text with another listing


## Cleaning the descriptions

The proposal said we'd remove names, phone numbers and emails from the descriptions. Regex works for emails and phone numbers. For names I first tried spaCy's small English model, but it removed 13,909 "names" that were mostly places and brands (St Kilda 713, WiFi 477, Albert Park 289, Fitzroy 276, Coles 130), and validation macro F1 dropped from 0.613 to 0.610. So it was deleting useful location words instead of personal details.

Instead I remove the host's own name from their description, using the host_name column before it gets dropped, plus any name that comes straight after "I'm", "I am", "my name is" or "hosted by". Words that appear in more than 1% of all descriptions aren't removed, because those are things like "Melbourne" in a company host name, not a person.

I keep the cleaning light on purpose. BERT needs normal sentences, so I don't lowercase, remove stopwords or stem. Week 2 also showed that removing stopwords can drop words like "not" and change the meaning.

In [ ]:
import html

from sklearn.feature_extraction.text import CountVectorizer

TAG = re.compile(r"<[^>]+>")
LINK = re.compile(r"https?://\S+|www\.\S+")
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE = re.compile(r"(?:\+?61|0)[\s-]?\d(?:[\s-]?\d){7,9}")
MONEY = re.compile(r"\$\s?\d[\d,]*(?:\.\d+)?")
# catches names in intros like "I'm Sarah" or "hosted by Tom"
INTRO = re.compile(r"\b(?:I['’]m|I am|[Mm]y name is|[Hh]osted by|[Cc]o-host)\s+([A-Z][a-z]+)")

# How many descriptions each capitalised word appears in. A real first name like
# "Sarah" is rare, but words in company host names like "Melbourne" or "Stays"
# are everywhere, so those shouldn't be removed.
counter = CountVectorizer(lowercase=False, binary=True, token_pattern=r"\b[A-Z][a-z]+\b")
doc_freq = dict(zip(counter.fit(desc).get_feature_names_out(),
                    np.asarray(counter.transform(desc).sum(axis=0)).ravel() / len(desc)))


def remove_names(text, host_name):
    removed = []
    words = re.findall(r"[A-Z][a-z]+", str(host_name)) if pd.notna(host_name) else []
    for w in words:
        if len(w) >= 3 and doc_freq.get(w, 0) < 0.01 and re.search(rf"\b{w}\b", text):
            removed.append(w)
            text = re.sub(rf"\b{w}\b", " ", text)
    for m in INTRO.finditer(text):
        removed.append(m.group(1))
    text = INTRO.sub(lambda m: m.group(0)[:-len(m.group(1))], text)
    return text, removed


# Each step builds on the one before, so I can measure what each one changes
stages = {"raw": desc}
s = desc.map(html.unescape).str.replace(TAG, " ", regex=True)
stages["html removed"] = s
s = s.str.replace(LINK, " ", regex=True).str.replace(EMAIL, " ", regex=True)
s = s.str.replace(PHONE, " ", regex=True)
stages["links, emails, phones removed"] = s
s = s.str.replace(MONEY, " ", regex=True)
stages["dollar amounts removed"] = s
results = [remove_names(t, h) for t, h in zip(s, df["host_name"])]
s = pd.Series([r[0] for r in results], index=s.index)
stages["names removed"] = s
s = s.str.replace(r"\s+", " ", regex=True).str.strip()
stages["whitespace tidied"] = s

removed_names = [w for r in results for w in r[1]]
print(f"Removed {len(removed_names):,} name mentions from "
      f"{sum(bool(r[1]) for r in results):,} descriptions")
# only counts, not which listing they came from, so I can check they look like names
print("Most common:", pd.Series(removed_names).value_counts().head(15).to_dict())
print("\nExample before and after:")
example = next(i for i, r in enumerate(results) if r[1])
print("BEFORE:", stages["dollar amounts removed"].iloc[example][:300])
print("AFTER: ", s.iloc[example][:300])

Removed 801 name mentions from 555 descriptions
Most common: {'Luxe': 43, 'Live': 35, 'Stays': 32, 'Urban': 29, 'Apartments': 24, 'Punthill': 19, 'Quest': 17, 'Abodes': 15, 'Artist': 15, 'Last': 14, 'Jar': 14, 'Taj': 14, 'Alix': 14, 'Hamptons': 13, 'Springs': 12}

Example before and after:
BEFORE: Sovereign Reserves – Great Stays ✨  Welcome to 620 Collins stylish and comfortable 2-bedroom, 1-bathroom apartment in Melbourne CBD. Enjoy a central location with Melbourne’s best dining, shopping, entertainment and attractions just moments away.  FREE PARKING!! 1 Car
AFTER:  – Great Stays ✨ Welcome to 620 Collins stylish and comfortable 2-bedroom, 1-bathroom apartment in Melbourne CBD. Enjoy a central location with Melbourne’s best dining, shopping, entertainment and attractions just moments away. FREE PARKING!! 1 Car


## Did the cleaning hurt anything?

Same idea as the Week 2 demo: a cleaning step is only worth keeping if it doesn't make the task worse. After each step I retrain a quick TF-IDF and logistic regression model and check macro F1 on the validation set. The test set isn't touched. If removing dollar amounts lowers the score, that means the amounts were leaking price information.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# retrain the same quick model after every step, scored on validation only
y = df["price_level"].cat.codes
is_train, is_val = df["split"] == "train", df["split"] == "val"
rows = []
for name, text in stages.items():
    tfidf = TfidfVectorizer(min_df=5, max_features=20000, ngram_range=(1, 2))
    X_train = tfidf.fit_transform(text[is_train])
    clf = LogisticRegression(max_iter=2000).fit(X_train, y[is_train])
    pred = clf.predict(tfidf.transform(text[is_val]))
    rows.append({"step": name,
                 "val macro F1": f1_score(y[is_val], pred, average="macro")})
cleaning_table = pd.DataFrame(rows).set_index("step").round(3)
print(cleaning_table)

                               val macro F1
step                                       
raw                                   0.614
html removed                          0.613
links, emails, phones removed         0.612
dollar amounts removed                0.613
names removed                         0.614
whitespace tidied                     0.614


## Missing data per modality

This is the completeness check from Week 2, done per split. I also check whether missing photos happen more for some price levels, because if they do, how we handle a missing photo could bias the model.

In [ ]:
df["description_clean"] = stages["whitespace tidied"]
df["has_photo"] = df["id"].isin(photo_manifest.loc[photo_manifest["ok"], "id"])
df["has_description"] = df["description_clean"] != ""

quality = pd.DataFrame({
    "photo": 1 - df.groupby("split")["has_photo"].mean(),
    "description": 1 - df.groupby("split")["has_description"].mean(),
    "review score": df.groupby("split")["review_scores_rating"].apply(lambda s: s.isna().mean()),
    "bedrooms": df.groupby("split")["bedrooms"].apply(lambda s: s.isna().mean()),
    "bathrooms": df.groupby("split")["bathrooms_text"].apply(lambda s: s.isna().mean()),
}).loc[["train", "val", "test"]]
print("Share missing, by split:")
print(quality.round(3))

# if missing photos aren't spread evenly across price levels, how we handle
# them could bias the model
print("\nShare without a photo, by price level:")
print((1 - df.groupby("price_level", observed=True)["has_photo"].mean()).round(3))

Share missing, by split:
       photo  description  review score  bedrooms  bathrooms
split                                                       
train  0.011        0.004         0.149     0.001      0.001
val    0.008        0.003         0.152     0.000      0.001
test   0.012        0.002         0.137     0.002      0.001

Share without a photo, by price level:
price_level
budget       0.007
mid-range    0.013
premium      0.011
Name: has_photo, dtype: float64


## Saving

The clean table goes to Drive. split_ids.csv (listing id and split) and target_edges.json (cap and price edges) are small and have no personal data, so they go on GitHub. Anyone rerunning the project can then check they got the same split and labels.

In [ ]:
# the original description isn't needed any more, the cleaned one replaces it.
# The group number was only for the split and host_name only for removing names.
out = df.drop(columns=["description", "group", "host_name"])
try:
    out.to_parquet(DATA_DIR / "listings_clean.parquet", index=False)
    out[["id", "split"]].to_csv(RESULTS_DIR / "split_ids.csv", index=False)
    cleaning_table.to_csv(RESULTS_DIR / "text_cleaning_check.csv")
except OSError as e:
    raise RuntimeError(f"Couldn't save the outputs: {e}")
print(f"Saved {len(out):,} listings and {out.shape[1]} columns")
print(list(out.columns))

Saved 18,808 listings and 17 columns
['id', 'price', 'room_type', 'property_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms_text', 'amenities', 'neighbourhood_cleansed', 'number_of_reviews', 'review_scores_rating', 'split', 'price_level', 'description_clean', 'has_photo', 'has_description']


## Making sure everything is saved

Colab uploads to Drive in the background, so this waits until the saved files are actually on Drive before the runtime closes.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Everything saved to Drive")

Everything saved to Drive
